# Bölüm 8: Tokenizer Oluşturma

> "Dil düşüncenin elbisesidir." — **Samuel Johnson**, Yazar

---

## Tokenizasyon Nedir?

**Tokenizasyon** metni sayılara dönüştürme işlemidir, böylece sinir ağları onu işleyebilir. Bunu İngilizce'yi gizli bir koda çevirmek olarak düşünün; her kelime, alt kelime veya karakter benzersiz bir sayı alır.

```
"Hello world" → [15496, 995]  (GPT-2'nin tokenizer'ını kullanarak)
```

Tokenizer tersi yönde de çalışır: sayılar verildiğinde metin üretir.

---

## Ne Öğreneceksiniz

- Üç farklı tokenizasyon stratejisiyle metnin nasıl sayılara dönüştüğü
- Karakter düzeyinde tokenizasyon neden basit ama verimsiz
- Kelime düzeyinde tokenizasyon bilinmeyen kelimeleri nasıl işler ve kelime dağarcığı boyutu neden önemli
- Modern LLM'lere güç veren alt kelime tokenizasyonunun (BPE) arkasındaki akıllı hile
- tiktoken ve Hugging Face transformers gibi üretim tokenizer'larını nasıl kullanılır
- Prompt'ları ve API maliyetlerini etkileyen tuhaflıklar ve yakalayıcılar

---

## Kurulum

İlk olarak, gerekli paketleri kuralım:

In [ ]:
# Gerekli paketleri kur
!pip install -q tiktoken transformers torch

## 1. Karakter Düzeyinde Tokenizasyon

En basit tokenizer'ı oluşturalım: her karakteri bir token olarak ele al.

**Python Sınıf Hatırlatması:**
- Bir **sınıf (class)** veri ve fonksiyonları bir araya getiren bir taslaktır
- `__init__(self)` bir nesne oluşturduğunuzda çalışır (verisini başlatır)
- `self` "bu belirli nesne"yi ifade eder ("bu araba" vs "genel olarak arabalar" gibi)
- `@property` bir metodu öznitelik gibi davranmasını sağlar (parantez gerekmez)

In [ ]:
class CharTokenizer:
    def __init__(self):
        # İki yönlü arama için iki sözlük
        self.char_to_id = {}
        self.id_to_char = {}

    def fit(self, text):
        """Metinden kelime dağarcığı oluştur.
        
        Neden sorted()? Böylece kelime dağarcığı deterministik olur - aynı metin
        her zaman aynı kimlikleri üretir. sorted() olmadan Python'un küme sırası rastgeledir.
        """
        # Benzersiz karakterleri çıkar - kümeler otomatik olarak benzersizliği işler
        chars = sorted(set(text))
        
        # Her karaktere bir kimlik ata (0'dan başlayarak)
        for i, c in enumerate(chars):
            self.char_to_id[c] = i
            self.id_to_char[i] = c

    def encode(self, text):
        """Metni token kimliklerinin listesine dönüştür.
        
        Karakter başına bir tamsayı listesi döndürür.
        """
        return [self.char_to_id[c] for c in text]

    def decode(self, ids):
        """Token kimliklerinin listesini tekrar metne dönüştür.
        
        Karakterleri aralarında boşluk olmadan birleştirmek için str.join() kullanır
        (kelime tokenizasyonundan farklı olarak boşluk gerekir).
        """
        return "".join(self.id_to_char[i] for i in ids)

    @property
    def vocab_size(self):
        """Kaç benzersiz karakter biliyoruz."""
        return len(self.char_to_id)

### Deneyin: Eksiksiz Örnek

In [ ]:
# Tokenizer oluştur ve eğit
tokenizer = CharTokenizer()
tokenizer.fit("Hello, world!")

print(f"Kelime dağarcığı boyutu: {tokenizer.vocab_size}")
print(f"Kelime dağarcığı: {tokenizer.char_to_id}")

# Kodla
text = "Hello"
ids = tokenizer.encode(text)
print(f"\n'{text}' → {ids}")

# Kod çöz - gidiş-dönüş kayıpsız olmalı!
decoded = tokenizer.decode(ids)
print(f"{ids} → '{decoded}'")
print(f"Mükemmel gidiş-dönüş? {text == decoded}")

### Kelime Dağarcığı Boyutu vs Dizi Uzunluğu Dengesi

In [ ]:
sentence = "Tokenization is the first step in any language model."

# Karakter düzeyi
char_tok = CharTokenizer()
char_tok.fit(sentence)
char_ids = char_tok.encode(sentence)

print(f"Orijinal metin: {len(sentence)} karakter")
print(f"Kelime dağarcığı boyutu: {char_tok.vocab_size}")
print(f"Dizi uzunluğu: {len(char_ids)}")
print(f"İlk 20 token: {char_ids[:20]}")

## 2. Kelime Düzeyinde Tokenizasyon

Şimdi özel token'larla bilinmeyen kelimeleri işleyen kelime düzeyinde bir tokenizer oluşturalım.

**Özel Token'lar:** Belirli anlamları olan ayrılmış kimlikler:
- `<PAD>` (Kimlik 0): Dolgu — dizileri yığınlama için eşit uzunluğa doldurur
- `<UNK>` (Kimlik 1): Bilinmeyen — kelime dağarcığımızda olmayan kelimeleri temsil eder
- `<BOS>` (Kimlik 2): Dizi Başı — metnin nerede başladığını işaretler
- `<EOS>` (Kimlik 3): Dizi Sonu — metnin nerede bittiğini işaretler

Bunlara neden ihtiyacımız var? `<UNK>` olmadan tokenizer'ımız yeni kelimeler karşısında çökerdi. `<PAD>` olmadan birden fazla cümleyi aynı anda işleyemezdik (farklı uzunluklara sahip olacaklardı).

In [ ]:
from collections import Counter

class WordTokenizer:
    def __init__(self, max_vocab_size=10000):
        """
        max_vocab_size: Maksimum kelime dağarcığı boyutu (özel token'lar dahil)
        
        Neden 10.000? Bu bir dengedir:
        - Çok küçük (1.000): Çok fazla bilinmeyen
        - Çok büyük (100.000): Devasa gömme tablosu, yavaş eğitim
        - 10.000-50.000: Öğrenme için ideal nokta
        """
        self.max_vocab_size = max_vocab_size
        self.word_to_id = {}
        self.id_to_word = {}

    def fit(self, text):
        """En sık kullanılan kelimelerden kelime dağarcığı oluştur."""
        # Basit başla: boşluklardan böl ve küçük harfe dönüştür
        words = text.lower().split()
        
        # Kelime sıklıklarını say - neden? Yaygın kelimeler kendi kimliklerini alır,
        # nadir kelimeler <UNK> olur. Bu pratikte bilinmeyenleri minimize eder.
        counts = Counter(words)
        
        # Özel token'lar için 0-3 kimliklerini ayır
        self.word_to_id = {
            "<PAD>": 0,
            "<UNK>": 1,
            "<BOS>": 2,
            "<EOS>": 3
        }
        self.id_to_word = {v: k for k, v in self.word_to_id.items()}
        
        # En yaygın kelimeleri ekle (max_vocab_size sınırını koruyarak)
        # 0-3 özel token'lar için ayrıldığından kimlikleri 4'ten başlat
        for i, (word, count) in enumerate(counts.most_common(self.max_vocab_size - 4), start=4):
            self.word_to_id[word] = i
            self.id_to_word[i] = word

    def encode(self, text, add_special_tokens=False):
        """Metni token kimliklerine dönüştür.
        
        add_special_tokens: True ise başa <BOS> ve sona <EOS> ekle
        """
        words = text.lower().split()
        
        # Her kelimeyi ara, kelime dağarcığında değilse <UNK>'ye (kimlik=1) geri dön
        # .get(word, 1) kelime kelime dağarcığımızda değilse 1 döndürür
        ids = [self.word_to_id.get(w, 1) for w in words]
        
        if add_special_tokens:
            ids = [2] + ids + [3]  # [<BOS>] + metin + [<EOS>]
        
        return ids

    def decode(self, ids, skip_special_tokens=True):
        """Token kimliklerini tekrar metne dönüştür.
        
        skip_special_tokens: True ise <PAD>, <BOS>, vb. çıktılama
        Neden? "<BOS> Hello world <EOS>" gibi çıktı istemezsiniz
        """
        words = []
        for i in ids:
            word = self.id_to_word.get(i, "<UNK>")
            # İstenirse çıktıda özel token'ları atla
            if skip_special_tokens and word in ["<PAD>", "<BOS>", "<EOS>"]:
                continue
            words.append(word)
        
        # Boşluklarla birleştir (karakter tokenizer'dan farklı olarak "".join kullandı)
        return " ".join(words)

    @property
    def vocab_size(self):
        """Mevcut kelime dağarcığı boyutu (benzersiz token sayısı)."""
        return len(self.word_to_id)

### Deneyin: Bilinmeyen Kelimelerle Eksiksiz Örnek

In [ ]:
# Bilinmeyenleri zorlamak için küçük kelime dağarcığıyla tokenizer oluştur
tokenizer = WordTokenizer(max_vocab_size=10)

# Sınırlı metinle eğit
training_text = """
The cat sat on the mat.
The cat was on the mat.
The dog sat on the mat.
"""
tokenizer.fit(training_text)

print(f"Kelime dağarcığı: {tokenizer.word_to_id}")

# Bilinen kelimelerle bir cümleyi kodla
text1 = "the cat sat"
ids1 = tokenizer.encode(text1)
print(f"\n'{text1}' → {ids1}")
print(f"Kod çözülmüş: '{tokenizer.decode(ids1)}'")

# Bilinmeyen kelimeyle kodla
text2 = "the elephant sat"  # "elephant" kelime dağarcığında değil!
ids2 = tokenizer.encode(text2)
print(f"\n'{text2}' → {ids2}")
print(f"Kod çözülmüş: '{tokenizer.decode(ids2)}'")

# Özel token'larla dene
ids3 = tokenizer.encode("the cat", add_special_tokens=True)
print(f"\nÖzel token'larla: {ids3}")
print(f"Kod çözülmüş (özel gösteriliyor): '{tokenizer.decode(ids3, skip_special_tokens=False)}'")
print(f"Kod çözülmüş (özel gizleniyor): '{tokenizer.decode(ids3, skip_special_tokens=True)}'")

### Karşılaştırma Egzersizi: Dengeyi Görün

In [ ]:
text = "The quick brown fox jumps over the lazy dog"

# Karakter tokenizer
char_tok = CharTokenizer()
char_tok.fit(text)
char_ids = char_tok.encode(text)

# Kelime tokenizer
word_tok = WordTokenizer(max_vocab_size=20)
word_tok.fit(text)
word_ids = word_tok.encode(text)

print("KARAKTER TOKENIZER:")
print(f"  Kelime dağarcığı boyutu: {char_tok.vocab_size}")
print(f"  Dizi uzunluğu: {len(char_ids)}")
print(f"  Token'lar: {char_ids[:20]}...")

print("\nKELİME TOKENIZER:")
print(f"  Kelime dağarcığı boyutu: {word_tok.vocab_size}")
print(f"  Dizi uzunluğu: {len(word_ids)}")
print(f"  Token'lar: {word_ids}")

## Üretim Tokenizer'ları Nasıl Çalışır: BPE (Byte-Pair Encoding)

Modern LLM'ler **alt kelime tokenizasyonu** kullanır — karakterler ve kelimeler arasında akıllı bir orta yol:

**Problem:**
- Karakter tokenizer'ları: Metin başına çok fazla token (yavaş, pahalı)
- Kelime tokenizer'ları: Yeni kelimeleri işleyemez ("ChatGPT" → `<UNK>`)

**Çözüm: BPE (Byte-Pair Encoding)**

BPE, en sık kullanılan karakter çiftlerini tekrar tekrar birleştirerek alt kelimeleri otomatik olarak öğrenir:

```
Adım 1: Karakterlerle başla: ["l", "o", "w", "e", "r"]
Adım 2: En sık çift ("l", "o") → "lo"ya birleştir
Adım 3: En sık çift ("lo", "w") → "low"a birleştir
Adım 4: Kelime dağarcığı boyutuna ulaşılana kadar devam et...
```

**Sonuç:** Yaygın kelimeler tek token olur, nadir kelimeler bilinen parçalara bölünür:
- "lower" → ["low", "er"] ✓ (yaygın, verimli)
- "lowest" → ["low", "est"] ✓ (bileşik kelime işlendi!)
- "ChatGPT" → ["Chat", "G", "PT"] ✓ (`<UNK>` gerekmez!)

**Ana Fikir:** BPE asla `<UNK>` üretmez çünkü herhangi bir karakter dizisi bilinen parçalara ayrılabilir!

---

## 3. Üretim Tokenizer'ları: tiktoken

Şimdi GPT-4'ün tokenizer'ı için OpenAI'nin tiktoken kütüphanesini kullanalım.

## Kendi BPE Tokenizer'ınızı Eğitme

Şimdi BPE'nin çalışırken öğrenmesini görelim! HuggingFace `tokenizers` kütüphanesini kullanarak Shakespeare üzerinde kendi tokenizer'ımızı eğiteceğiz.

**Bunu neden yapmalı?**
- Kalıpların otomatik olarak ortaya çıktığını görün ("ing", "the", "tion")
- Tokenizer'ların nasıl çalıştığına dair sezgi oluşturun
- Alan uyuşmazlığını anlayın (Shakespeare tokenizer vs modern metin)

İlk olarak tokenizers kütüphanesini kuralım:

In [ ]:
# tokenizers kütüphanesini kur (transformers'tan ayrı!)
!pip install -q tokenizers

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

print("İçe aktarmalar başarılı!")

### TinyShakespeare'i İndir

Bölüm 12'de tam LLM'inizi eğiteceğiniz aynı veri kümesini kullanacağız:

In [ ]:
import urllib.request

# TinyShakespeare'i indir (Bölüm 12'de kullanacağımız veri kümesi!)
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "shakespeare.txt")

# Oku ve incele
with open("shakespeare.txt", "r") as f:
    text = f.read()

print(f"Veri kümesi boyutu: {len(text):,} karakter")
print(f"\nİlk 200 karakter:")
print(text[:200])

### BPE Tokenizer'ı Eğit

Şimdi heyecan verici kısım - BPE'nin Shakespeare'den kalıplar öğrenmesini izleyin!

In [ ]:
# Boş bir BPE tokenizer oluştur
bpe_tokenizer = Tokenizer(BPE(unk_token="<UNK>"))

# BPE birleştirmesinden önce boşluklara böl
bpe_tokenizer.pre_tokenizer = Whitespace()

# Eğiticisi yapılandır
trainer = BpeTrainer(
    vocab_size=1000,           # Öğrenmek için küçük (üretim 30.000+ kullanır)
    min_frequency=2,           # Token en az iki kez görünmeli
    special_tokens=["<PAD>", "<UNK>", "<BOS>", "<EOS>"]
)

# Shakespeare üzerinde eğit (bu sadece birkaç saniye alır!)
bpe_tokenizer.train(files=["shakespeare.txt"], trainer=trainer)

print(f"Kelime dağarcığı boyutu: {bpe_tokenizer.get_vocab_size()}")

### Eğitilmiş Tokenizer'ınızı Test Edin

In [ ]:
# Shakespeare tarzı metinde test et
test_text = "To be or not to be, that is the question."
output = bpe_tokenizer.encode(test_text)

print(f"Metin: '{test_text}'")
print(f"Token'lar: {output.tokens}")
print(f"Token Kimlikleri: {output.ids}")
print(f"Token sayısı: {len(output.ids)}")

# Gidiş-dönüşü doğrula
decoded = bpe_tokenizer.decode(output.ids)
print(f"\nKod çözülmüş: '{decoded}'")
print(f"Mükemmel gidiş-dönüş: {test_text == decoded}")

### Alan Uyuşmazlığı: Shakespeare Tokenizer'ında Modern Metin

Shakespeare'in asla yazmadığı modern teknik metinde ne olur?

In [ ]:
# Shakespeare'in asla yazmadığı modern metni dene
modern_text = "ChatGPT generates amazing neural network responses"
output2 = bpe_tokenizer.encode(modern_text)

print(f"Metin: '{modern_text}'")
print(f"Token'lar: {output2.tokens}")
print(f"Token sayısı: {len(output2.ids)}")

# Dikkat: "ing" bir token'dır (Shakespeare kullandı!) ama "ChatGPT" parçalara ayrılır
print("\nAna fikir:")
print("- 'ing' tek bir token'dır (Shakespeare '-ing' kelimelerini sık kullandı)")
print("- 'ChatGPT' parçalara ayrılır (Shakespeare AI hakkında yazmadı!)")
print("- Bu alan uyuşmazlığının pratikte halidir")

---

## Üretim Tokenizer'ları: tiktoken

Artık BPE'nin sıfırdan nasıl öğrendiğini gördünüz. O üretim tokenizer'larının içinde ne olduğunu anlıyorsunuz. tiktoken "ChatGPT"yi parçalara ayırdığında nedenini biliyorsunuz!

Projelerinizde gerçekten kullanacağınız üretim araçlarını kullanalım.

In [ ]:
import tiktoken

# GPT-4'ün tokenizer kodlamasını yükle
enc = tiktoken.get_encoding("cl100k_base")  # GPT-4, GPT-3.5-turbo

# Metni kodla
text = "Hello, world! How are you?"
tokens = enc.encode(text)

print(f"Metin: '{text}'")
print(f"Token'lar: {tokens}")
print(f"Token sayısı: {len(tokens)}")

# Geri kod çöz
decoded = enc.decode(tokens)
print(f"Kod çözülmüş: '{decoded}'")
print(f"Mükemmel gidiş-dönüş: {text == decoded}")

### Gerçek Token String'lerini Görün

In [ ]:
# Her token'ı ayrı ayrı kod çözerek neyi temsil ettiğini gör
token_strings = [enc.decode([t]) for t in tokens]

print(f"Token dökümü:")
for token_id, token_str in zip(tokens, token_strings):
    print(f"  {token_id:5d} → '{token_str}'")

### API Bütçelemesi için Token Sayımı

In [ ]:
prompts = [
    "Write a haiku about programming.",
    "Explain quantum computing in simple terms for a 10-year-old child.",
    "Generate a 500-word essay on climate change."
]

enc = tiktoken.get_encoding("cl100k_base")

for prompt in prompts:
    tokens = enc.encode(prompt)
    # Örnek fiyatlandırma (güncel oranları openai.com/api/pricing'de kontrol edin)
    cost = len(tokens) * 0.00001  # 1K girdi token başına $0.01
    
    print(f"Prompt: '{prompt}'")
    print(f"  Token'lar: {len(tokens)}")
    print(f"  Maliyet (girdi): ~${cost:.5f}\n")

## 4. Üretim Tokenizer'ları: Hugging Face Transformers

**Hugging Face** önceden eğitilmiş modeller ve tokenizer'lar sağlayan bir şirket/kütüphanedir. `AutoTokenizer` herhangi bir model için doğru tokenizer'ı otomatik olarak yükler.

Hugging Face tokenizer şunları içeren bir sözlük döndürür:
- `input_ids`: Token kimlikleri (en çok önemsediğimiz)
- `attention_mask`: Gerçek token'lar için 1'ler, dolgu için 0'lar (modele neyi görmezden geleceğini söyler)

In [ ]:
from transformers import AutoTokenizer

# GPT-2 tokenizer'ı yükle (açık kaynak)
tokenizer = AutoTokenizer.from_pretrained("gpt2")

text = "Hello, world! How are you?"

# Kodla - token kimlikleri ve dikkat maskesi ile sözlük döndürür
encoded = tokenizer(text, return_tensors="pt")  # "pt" = PyTorch tensörleri

print(f"Metin: '{text}'")
print(f"Token Kimlikleri: {encoded['input_ids']}")
print(f"Dikkat maskesi: {encoded['attention_mask']}")

# Kod çöz
decoded = tokenizer.decode(encoded['input_ids'][0])
print(f"Kod çözülmüş: '{decoded}'")

### Token Dökümü

In [ ]:
tokens = encoded['input_ids'][0].tolist()
token_strings = [tokenizer.decode([t]) for t in tokens]

print(f"Token dökümü:")
for tid, tstr in zip(tokens, token_strings):
    print(f"  {tid:5d} → '{tstr}'")

## 5. Tokenizasyon Tuhaf Durumları

### Tuhaflık #1: Başındaki Boşluklar Her Şeyi Değiştirir

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

# Başında boşluk olan ve olmayan ile karşılaştır
texts = ["Hello", " Hello", "world", " world"]

for text in texts:
    tokens = enc.encode(text)
    print(f"'{text}' → {tokens} ({len(tokens)} token{'s' if len(tokens) > 1 else ''})")

### Tuhaflık #2: Sayılar Rakamlara Göre Bölünür

In [ ]:
numbers = ["10", "100", "1000", "10000", "42", "2024"]

enc = tiktoken.get_encoding("cl100k_base")

for num in numbers:
    tokens = enc.encode(num)
    token_strs = [enc.decode([t]) for t in tokens]
    print(f"'{num}' → {tokens} = {token_strs}")

### Tuhaflık #3: Emoji ve Özel Karakterler

In [ ]:
emojis = ["😀", "🚀", "👍", "Hello 😀 world", "🔥🔥🔥"]

enc = tiktoken.get_encoding("cl100k_base")

for text in emojis:
    tokens = enc.encode(text)
    token_strs = [enc.decode([t]) for t in tokens]
    print(f"'{text}' → {len(tokens)} token: {token_strs}")

## 6. Pratik Egzersiz: Veri Kümenizi Tokenize Edin

Bölüm 7'nin veri kümesini tokenizasyona bağlayın (bunun için chapter7_output.jsonl dosyanıza ihtiyacınız olacak):

In [ ]:
import json
import tiktoken

# Örnek veri kümesi (Bölüm 7 çıktınızla değiştirin)
example_dataset = [
    {"text": "AI systems learn from examples", "split": "train"},
    {"text": "Neural networks need lots of data", "split": "train"},
    {"text": "Deep learning uses multiple layers", "split": "val"}
]

# Her örneği tokenize et
enc = tiktoken.get_encoding("cl100k_base")

for record in example_dataset:
    text = record["text"]
    tokens = enc.encode(text)
    record["token_ids"] = tokens
    record["token_count"] = len(tokens)

print(f"{len(example_dataset)} örnek tokenize edildi")

# İstatistikleri hesapla
token_counts = [r["token_count"] for r in example_dataset]
avg_tokens = sum(token_counts) / len(token_counts)
max_tokens = max(token_counts)
min_tokens = min(token_counts)

print(f"\nİstatistikler:")
print(f"  Örnek başına ortalama token: {avg_tokens:.1f}")
print(f"  Maksimum token: {max_tokens}")
print(f"  Minimum token: {min_tokens}")

# İlk örneği göster
print(f"\nİlk örnek:")
print(json.dumps(example_dataset[0], indent=2))

## Bölüm Özeti

**Ne oluşturduk:**

1. **Karakter tokenizer:** Basit ama verimsiz (küçük kelime dağarcığı ~100, uzun diziler)
2. **Kelime tokenizer:** Verimli diziler ama devasa kelime dağarcığı ve bilinmeyen kelime sorunları
3. **BPE tokenizer:** Shakespeare üzerinde kendinizinkini eğittiniz, kalıpların otomatik olarak ortaya çıkışını gördünüz!
4. **Üretim araçları:** Gerçek dünya tokenizasyonu için tiktoken ve Hugging Face kullandınız

**Ne öğrendik:**

- Tokenizasyon tersine çevrilebilir (kayıpsız gidiş-dönüş)
- Kelime dağarcığı boyutu vs dizi uzunluğu dengesi temeldir
- Özel token'lar belirli amaçlara hizmet eder (BOS/EOS/PAD/UNK)
- BPE sık çiftleri birleştirerek alt kelimeleri otomatik olarak öğrenir
- Alan uyuşmazlığı önemlidir: Shakespeare tokenizer modern teknoloji kelimeleriyle zorlanır
- Tokenizasyonun tuhaf durumları vardır (başındaki boşluklar, sayı bölme, emoji)

**Sırada:** Bölüm 9 bu token kimliklerini gömme vektörlerine dönüştürecek!